In [1]:
#Universal Sentence Encoder (USE) - DAN

import pandas as pd
import re
from nltk.corpus import stopwords
import tensorflow_hub as hub
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

# Stopwords setini indir
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yusuf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Remove special characters, punctuation, and convert to lowercase
def preprocess_text(text):
    if not isinstance(text, str):
        return ''  # Return empty string for NaN or non-string values
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)  # Remove stopwords
    return text

In [3]:
# Load the dataset
file_path = "filtered_file_2.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Apply text preprocessing
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

# Combine text columns for analysis
df['combined_text'] = df['Abstract_clean'] + ' ' + df['Research_Areas_clean']


In [4]:
# User query
user_input = "cell cancer bio bioinformatics machine learning algorithm human life gene gen heredity"
user_input_clean = preprocess_text(user_input)

# Load Universal Sentence Encoder (DAN)
use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

# Encode dataset and user query
print("Encoding dataset...")
corpus_embeddings = use_model(df['combined_text'].tolist())  # Encode all texts
query_embedding = use_model([user_input_clean])  # Encode user query

# Compute cosine similarity
print("Computing similarity scores...")
similarities = cosine_similarity(query_embedding, corpus_embeddings).flatten()

# Add similarity scores to the dataframe
df['similarity'] = similarities

# Calculate article count per researcher
df['article_count'] = df['Researcher Ids'].str.split(';').apply(len)

Encoding dataset...
Computing similarity scores...


In [5]:
# Flatten researchers for grouping
def split_researchers(row):
    if not isinstance(row['Researcher Ids'], str):
        return pd.DataFrame()
    researchers = row['Researcher Ids'].split(';')
    data = []
    for researcher in researchers:
        data.append({
            'Researcher': researcher.strip(),
            'combined_text': row['combined_text'],
            'Publication Year': row['Publication Year'],
            'similarity': row['similarity'],
            'article_count': row['article_count']
        })
    return pd.DataFrame(data)

flattened_df = pd.concat(df.apply(split_researchers, axis=1).to_list(), ignore_index=True)

# Group by researcher and calculate average similarity
grouped_df = flattened_df.groupby('Researcher', as_index=False).agg({
    'similarity': 'mean',
    'article_count': 'sum',
    'Publication Year': 'max'
})

# Count the number of articles per researcher
article_counts = flattened_df['Researcher'].value_counts()
grouped_df['article_count'] = grouped_df['Researcher'].map(article_counts)

# Scale article count weight
optimal_point = 25
sigma = 20
grouped_df['article_weight'] = 1 / (1 + ((grouped_df['article_count'] - optimal_point) ** 2) / sigma**2)

# Normalize similarity by article weight
grouped_df['normalized_similarity'] = grouped_df['similarity'] * grouped_df['article_weight']

# Scale year weights for stronger impact (higher weight for recent years)
current_year = 2025
year_scaling_factor = 1.5
grouped_df['year_weight'] = 1 / (1 + (np.log(1 + current_year - grouped_df['Publication Year']) / np.log(1000)) * year_scaling_factor)

# Compute final score
grouped_df['final_score'] = grouped_df['normalized_similarity'] * grouped_df['year_weight']

# Get top researchers based on final score
top_researchers = grouped_df.sort_values(by='final_score', ascending=False).head(30)
print(top_researchers[['Researcher', 'final_score', 'article_count', 'year_weight', 'Publication Year']])

                                  Researcher  final_score  article_count  \
7951              Işık, Zerrin/ABG-9392-2021     0.324795             22   
14242             Senturk, Serif/A-1263-2017     0.311818             19   
6463   Georgakilas, Alexandros/AAG-9517-2020     0.305857             20   
3247             Caglayan, Ahmet/D-1066-2012     0.301909             26   
8659                Karaca, Ezgi/P-1759-2019     0.298998             27   
9343                Koc, Altug/AAH-3159-2020     0.296329             25   
13644                SUNER, ASLI/C-1032-2012     0.290470             24   
3294       Calibasi Kocal, Gizem/G-1761-2014     0.283370             32   
15649        Tufekci, Kemal Ugur/A-2850-2014     0.281464             30   
9633                  Kucuk, Can/K-3913-2015     0.280619             16   
19154             sercan, ogun/JFA-0026-2023     0.270880             15   
18594             icsel, ceyda/AAI-3342-2021     0.265871             18   
12230       